# DenseArraySequencer Basics

This tutorial introduces `DenseArraySequencer`, which wraps `AkwardJaggedAutoregressiveSequencer` for dense (single time-series) arrays. It produces sliding input/output windows for autoregressive forecasting.

## Autoregressive Windowing

For each anchor time step `t`:

- **seq_x (input)**: `[t - seq_len : t]` — encoder history window
- **seq_y (target)**: `[t + pred_offset : t + pred_offset + label_len + pred_len]` — decoder overlap + forecast horizon

```
Time:    0  1  2  3  4  5  6  7  8  9  ...
         |<-- seq_len -->|<- label_len ->|<- pred_len ->|
         [   seq_x   ]   [ overlap ]     [  forecast  ]
                         [<------ seq_y ------------>]
```

The array must have at least `seq_len + pred_len + pred_offset` elements per unit.

## Stride and warmup_steps

- **stride**: Step size between consecutive windows. `stride=2` means windows start at indices 0, 2, 4, ... (or according to warmup). Fewer windows, less overlap.
- **warmup_steps**: With left padding enabled (default), controls how many padded steps appear in the first window. Default `seq_len - 1` maximizes padding so the first window contains only 1 real observation at the end.

In [ ]:
import numpy as np
from picid.data.optimization.sequencer import DenseArraySequencer

arr = np.random.randn(100, 3).astype(np.float32)
seq = DenseArraySequencer(
    array=arr,
    seq_len=4,
    label_len=2,
    pred_len=2,
    stride=2,
)
print(f"Total windows: {len(seq)}")

In [ ]:
batch = seq.sequences_batch([0, 1, 2])
seq_x, seq_y = batch[0], batch[1]
print(f"seq_x: {seq_x.shape}, seq_y: {seq_y.shape}")
assert seq_x.shape == (3, 4, 3), f"Expected (3, 4, 3), got {seq_x.shape}"
assert seq_y.shape == (3, 4, 3), f"Expected (3, 4, 3), got {seq_y.shape}"
print("OK")

## Summary

- `seq_x` has shape `(batch, seq_len, num_features)` — encoder input
- `seq_y` has shape `(batch, label_len + pred_len, num_features)` — decoder target (overlap + forecast)
- Use `get_index_array()` to see valid window indices; `sequences_batch(idx)` fetches a batch by those indices.